# 孪生网络


## Colab 准备


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from os import path

import numpy as np
import random

%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt

import torch
from torch.optim import lr_scheduler
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.datasets import MNIST
from torchvision import transforms


# 1. 设置与初始化
我们将在 MNIST 数据集上用不同的损失函数来学习特征 embedding。这只是为了可视化，所以我们用 2 维 embedding，这在实践中并不是最好的选择。

每个实验使用相同的 embedding 网络（`32 conv 5x5 -> ReLU -> MaxPool 2x2 -> 64 conv 5x5 -> ReLU -> MaxPool 2x2 -> 全连接 256 -> ReLU -> 全连接 256 -> ReLU -> 全连接 2`）和相同的超参数。


In [ ]:
class ExperimentParams():
    def __init__(self):
        self.num_classes = 10
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.batch_size = 256
        self.lr = 1e-2
        self.num_epochs = 10
        self.num_workers = 4
        self.data_dir = '/home'
        

args = ExperimentParams()

## 1.1 准备数据集
我们将在 MNIST 数据集上工作


In [ ]:

mean, std = 0.1307, 0.3081

train_dataset = MNIST(f'{args.data_dir}/data/MNIST', train=True, download=True,
                             transform=transforms.Compose([
                                 transforms.ToTensor(),
                                 transforms.Normalize((mean,), (std,))
                             ]))

test_dataset = MNIST(f'{args.data_dir}/data/MNIST', train=False, download=True,
                            transform=transforms.Compose([
                                transforms.ToTensor(),
                                transforms.Normalize((mean,), (std,))
                            ]))

## 1.2 通用设置


In [ ]:

mnist_classes = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728',
              '#9467bd', '#8c564b', '#e377c2', '#7f7f7f',
              '#bcbd22', '#17becf']

def plot_embeddings(embeddings, targets, title='',xlim=None, ylim=None):
    plt.figure(figsize=(10,10))
    for i in range(10):
        inds = np.where(targets==i)[0]
        plt.scatter(embeddings[inds,0], embeddings[inds,1], alpha=0.5, color=colors[i])
    if xlim:
        plt.xlim(xlim[0], xlim[1])
    if ylim:
        plt.ylim(ylim[0], ylim[1])
    plt.legend(mnist_classes)
    plt.title(title)

def extract_embeddings(dataloader, model, args):
    with torch.no_grad():
        model.eval()
        embeddings = np.zeros((len(dataloader.dataset), 2))
        labels = np.zeros(len(dataloader.dataset))
        k = 0
        for images, target in dataloader:
            images = images.to(args.device)
            embeddings[k:k+len(images)] = model.get_embedding(images).data.cpu().numpy()
            labels[k:k+len(images)] = target.numpy()
            k += len(images)
    return embeddings, labels


def get_raw_images(dataloader,mean=0.1307, std=0.3081):

    raw_images = np.zeros((len(dataloader.dataset), 1, 28, 28))
    k = 0
    for input, target in dataloader:
        raw_images[k:k+len(input)] = (input*std + mean).data.cpu().numpy()
        k += len(input)

    return raw_images


def show(img, title=None):
    # img 是一个 torch.Tensor
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1,2,0)), interpolation='nearest')
    plt.axis('off')
    if title is not None:
        plt.title(title)
    plt.pause(0.001)  # 稍微暂停一下，让图像刷新出来

# 2. 基线：用 softmax 分类
我们将训练模型做分类，并把倒数第二层的输出作为 embedding。


我们将定义基础的 embedding 架构，作为我们实验的公共骨干网络


## 2.1 架构


### 练习

补全下面 `EmbeddingNet` 架构定义中缺失的部分：（`32 conv 5x5 -> ReLU -> MaxPool 2x2 -> 64 conv 5x5 -> ReLU -> MaxPool 2x2 -> 全连接 256 -> ReLU -> 全连接 256 -> ReLU -> 全连接 2`）


In [ ]:

class EmbeddingNet(nn.Module):
    def __init__(self):
        super(EmbeddingNet, self).__init__()
#         self.conv1 = nn.Conv2d(1, ...)
#         self.conv2 = ...
#         self.fc1 = ...
#         self.fc2 = ...
#         self.fc3 = ...

    def forward(self, x, debug=False):
        x1 = F.max_pool2d(F.relu(self.conv1(x)), kernel_size=2, stride=2)
#       output = ...
        if debug == True:
            print(f'input: {x.size()}')
            print(f'x1: {x1.size()}')
                      
        return output

    def get_embedding(self, x):
        return self.forward(x)

如果你想更好地检查隐状态的大小并做调试，可以在 `forward` 函数里加一个 `debug` 变量，就像上面那样


In [ ]:
input = torch.zeros(1, 1, 28, 28)
net = EmbeddingNet()
output = net(input,debug=True)
print(f'output size: {output.size()}')

### 问题
输出的维度是 `batch-size x 2`。为什么？


现在定义一个分类网络，在 `EmbeddingNet` 之上加全连接层


### 练习

填写 `forward` 中缺失的部分。


In [ ]:

class ClassificationNet(nn.Module):
    def __init__(self, embedding_net, num_classes):
        super(ClassificationNet, self).__init__()
        self.embedding_net = embedding_net
        self.fc = nn.Linear(2, num_classes)

    def forward(self, x, debug=False):
        # 把 None 换成需要的条目
        embedding = None
        output = F.softplus(self.fc(embedding))
        
        # if debug == True:
        #     print(f'input: {x.size()}')
        #     print(f'embedding: {embedding.size()}')
        #     print(f'output: {output.size()}')
            
        return output
    
    def get_embedding(self, x):
        # 把 None 换成需要的条目
        return None

## 2.2 训练


In [ ]:
# 设置数据加载器

kwargs = {'num_workers': args.num_workers, 'pin_memory': True} 
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True, **kwargs)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=args.batch_size, shuffle=False, **kwargs)

embedding_net = EmbeddingNet()
model = ClassificationNet(embedding_net, num_classes=args.num_classes)
loss_fn = torch.nn.CrossEntropyLoss()
model.to(args.device)
loss_fn.to(args.device)

optimizer = optim.Adam(model.parameters(), lr=args.lr)

In [ ]:
train_embeddings_baseline, train_labels_baseline = extract_embeddings(train_loader, model, args)
plot_embeddings(train_embeddings_baseline, train_labels_baseline, 'Train embeddings before training')

In [ ]:
def train_classif_epoch(train_loader, model, loss_fn, optimizer, args, log_interval=50):
    model.train()
    losses = []
    total_loss, total_corrects, num_samples = 0, 0, 0
    corrects = 0    
    for batch_idx, (data, target) in enumerate(train_loader):
        num_samples += data.size(0)
        
        data, target = data.to(args.device), target.to(args.device)
        
        optimizer.zero_grad()
        outputs = model(data)

        loss = loss_fn(outputs, target)
        losses.append(loss.data.item())

        _,preds = torch.max(outputs.data,1)
        corrects += torch.sum(preds == target.data).cpu()

        loss.backward()
        optimizer.step()
        
        if batch_idx % log_interval == 0:
            print('Train: [{}/{} ({:.0f}%)]\tLoss: {:.6f} \tAccuracy: {}'.format(
                batch_idx * len(data[0]), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), np.mean(losses), float(total_corrects)/num_samples))           
            
            total_loss += np.sum(losses)
            total_corrects += corrects
            losses, corrects = [], 0

    return total_loss/(batch_idx + 1), total_corrects/num_samples

def test_classif_epoch(test_loader, model, loss_fn, args, log_interval=50):
    with torch.no_grad():
        model.eval()
        losses, corrects = [], 0
        num_samples = 0
    
        for batch_idx, (data, target) in enumerate(test_loader):

            num_samples += data.size(0)
            data, target = data.to(args.device), target.to(args.device)

            outputs = model(data)

            loss = loss_fn(outputs, target)
            losses.append(loss.data.item())

            _,preds = torch.max(outputs.data,1)
            corrects += torch.sum(preds == target.data).cpu()

        return np.sum(losses)/(batch_idx + 1), corrects/num_samples

### 问题
为什么需要 `optimizer.zero_grad()`？如果去掉它会怎样？


In [ ]:
start_epoch = 0

for epoch in range(start_epoch, args.num_epochs):

    train_loss, train_accuracy = train_classif_epoch(train_loader, model, loss_fn, optimizer, args)

    message = 'Epoch: {}/{}. Train set: Average loss: {:.4f} Average accuracy: {:.4f}'.format(
        epoch + 1, args.num_epochs, train_loss, train_accuracy)
    
    val_loss, val_accuracy = test_classif_epoch(test_loader, model, loss_fn, args)
    
    message += '\nEpoch: {}/{}. Validation set: Average loss: {:.4f}  Average accuracy: {:.4f}'.format(epoch + 1, args.num_epochs,
                                                                             val_loss, val_accuracy)
    print(message)


## 2.3 可视化


In [ ]:
train_embeddings_baseline, train_labels_baseline = extract_embeddings(train_loader, model, args)
plot_embeddings(train_embeddings_baseline, train_labels_baseline, 'Train embeddings classification')
test_embeddings_baseline, test_labels_baseline = extract_embeddings(test_loader, model, args)
plot_embeddings(test_embeddings_baseline, test_labels_baseline, 'Test embeddings classification')

In [ ]:
train_embeddings_baseline, train_labels_baseline = extract_embeddings(train_loader, model, args)
plot_embeddings(train_embeddings_baseline, train_labels_baseline, 'Train embeddings classification')
test_embeddings_baseline, test_labels_baseline = extract_embeddings(test_loader, model, args)
plot_embeddings(test_embeddings_baseline, test_labels_baseline, 'Test embeddings classification')

虽然 embedding 看起来可分（这正是我们训练它的目的），但它们没有好的度量性质。作为新类别的描述子，它们可能不是最好的选择。


# 3. 孪生网络
现在我们将训练一个孪生网络：它接收一对图像，训练 embedding，使得如果它们来自同一类，距离被最小化；如果它们代表不同的类，距离大于某个边际值。
我们将最小化一个对比损失函数*：
$$L_{contrastive}(x_0, x_1, y) = \frac{1}{2} y \lVert f(x_0)-f(x_1)\rVert_2^2 + \frac{1}{2}(1-y)\{max(0, m-\lVert f(x_0)-f(x_1)\rVert_2)\}^2$$

*Raia Hadsell, Sumit Chopra, Yann LeCun, [Dimensionality reduction by learning an invariant mapping](http://yann.lecun.com/exdb/publis/pdf/hadsell-chopra-lecun-06.pdf), CVPR 2006*


## 3.1 架构
我们首先在 `EmbeddingNet` 之上定义孪生架构


### 练习

填写 `SiameseNet` 的 forward 部分


In [ ]:
class SiameseNet(nn.Module):
    def __init__(self, embedding_net):
        super(SiameseNet, self).__init__()
        self.embedding_net = embedding_net

    def forward(self, x1, x2):
        # 填写缺失的 2 行 :)
        output1 = self.embedding_net(x1)
        output2 = self.embedding_net(x2)
        
        return output1, output2

    def get_embedding(self, x):
        return self.embedding_net(x)

## 3.2 数据加载器
我们还需要调整数据加载器，让它取成对的图像


In [ ]:
from torch.utils.data import Dataset
from torch.utils.data.sampler import BatchSampler
from PIL import Image

class SiameseMNIST(Dataset):
    """
    train mode: For each sample creates randomly a positive or a negative pair
    test mode: Creates fixed pairs for testing
    """

    def __init__(self, mnist_dataset):
        self.mnist_dataset = mnist_dataset

        self.train = self.mnist_dataset.train
        self.transform = self.mnist_dataset.transform

        if self.train:
            self.train_labels = self.mnist_dataset.train_labels
            self.train_data = self.mnist_dataset.train_data
            self.labels_set = set(self.train_labels.numpy())
            self.label_to_indices = {label: np.where(self.train_labels.numpy() == label)[0]
                                     for label in self.labels_set}
        else:
            # 生成固定的测试用样本对
            self.test_labels = self.mnist_dataset.test_labels
            self.test_data = self.mnist_dataset.test_data
            self.labels_set = set(self.test_labels.numpy())
            '''
            create a dictionary with an entry key for each label and the value an array storing
            the indices of the images having the respective label
            '''
            self.label_to_indices = {label: np.where(self.test_labels.numpy() == label)[0]
                                     for label in self.labels_set}

            random_state = np.random.RandomState(42)
            # 遍历 test_data，随机选择相同标签的样本
            positive_pairs = [[i,
                               random_state.choice(self.label_to_indices[self.test_labels[i].item()]),
                               1]
                              for i in range(0, len(self.test_data), 2)]

            # 遍历 test_data，创建所有与当前标签不同的标签列表，然后
            # 随机选择具有这些标签之一的样本
            negative_pairs = [[i,
                               random_state.choice(self.label_to_indices[
                                                       np.random.choice(
                                                           list(self.labels_set - set([self.test_labels[i].item()]))
                                                       )
                                                   ]),
                               0]
                              for i in range(1, len(self.test_data), 2)]
            # 格式：[index1, index2, label(0/1)]
            self.test_pairs = positive_pairs + negative_pairs

    def __getitem__(self, index):
        
        # 训练时样本对随机、即时地获取
        if self.train:
            # 随机选择一个标签，即相似（1）或不相似（0）的图像
            target = np.random.randint(0, 2)
            img1, label1 = self.train_data[index], self.train_labels[index].item()
            if target == 1:
                # 选一张与 img1 标签相同的图像
                siamese_index = index
                while siamese_index == index:
                    siamese_index = np.random.choice(self.label_to_indices[label1])
            else:
                # 从可选的标签集合中去掉 label1
                siamese_label = np.random.choice(list(self.labels_set - set([label1])))
                # 从该子集中随机选择一张图像
                siamese_index = np.random.choice(self.label_to_indices[siamese_label])
            img2 = self.train_data[siamese_index]
        else:
            img1 = self.test_data[self.test_pairs[index][0]]
            img2 = self.test_data[self.test_pairs[index][1]]
            target = self.test_pairs[index][2]

        img1 = Image.fromarray(img1.numpy(), mode='L')
        img2 = Image.fromarray(img2.numpy(), mode='L')
        if self.transform is not None:
            img1 = self.transform(img1)
            img2 = self.transform(img2)
        return (img1, img2), target

    def __len__(self):
        return len(self.mnist_dataset)

## 3.3 损失函数


$$L_{contrastive}(x_0, x_1, y) = \frac{1}{2} y \lVert f(x_0)-f(x_1)\rVert_2^2 + \frac{1}{2}(1-y)\{max(0, m-\lVert f(x_0)-f(x_1)\rVert_2)\}^2$$


### 练习

填写 `contrastive loss` 中缺失的部分


In [ ]:
class ContrastiveLoss(nn.Module):
    """
    Contrastive loss
    Takes embeddings of two samples and a target label == 1 if samples are from the same class and label == 0 otherwise
    """

    def __init__(self, margin):
        super(ContrastiveLoss, self).__init__()
        self.margin = margin
        self.eps = 1e-9

    def forward(self, output1, output2, target, size_average=True):
        # 计算 output2 和 output1 之间的平方距离
        squared_distances = (output2 - output1).pow(2).sum(1)
        # 把损失的第二项加进去。你可以用 ReLU 来实现 max 公式
        # losses = 0.5 * (target.float() * squared_distances +
        #                  (1-target).float() * F.relu(self.margin - (squared_distances + self.eps).sqrt()).pow(2))
        
        losses = 0.5 * (target.float() * squared_distances +
                         (1-target).float() * F.relu(self.margin - squared_distances + self.eps))
        
        return losses.mean() if size_average else losses.sum()
        


## 3.4 训练


In [ ]:
# 设置数据加载器
siamese_train_dataset = SiameseMNIST(train_dataset)  # 返回图像对和目标（相同/不同）
siamese_test_dataset = SiameseMNIST(test_dataset)

args.batch_size = 128
kwargs = {'num_workers': args.num_workers, 'pin_memory': True}
siamese_train_loader = torch.utils.data.DataLoader(siamese_train_dataset, batch_size=args.batch_size, shuffle=True, **kwargs)
siamese_test_loader = torch.utils.data.DataLoader(siamese_test_dataset, batch_size=args.batch_size, shuffle=False, **kwargs)

margin = 1.
embedding_net = EmbeddingNet()
model = SiameseNet(embedding_net)
loss_fn = ContrastiveLoss(margin)
model.to(args.device)
loss_fn.to(args.device)

args.lr = 1e-3
optimizer = optim.Adam(model.parameters(), lr=args.lr)


In [ ]:
def train_siamese_epoch(train_loader, model, loss_fn, optimizer, args, log_interval=100):
    model.train()
    losses = []
    total_loss, num_samples =  0, 0
  
    for batch_idx, (data, target) in enumerate(train_loader):
        num_samples += data[0].size(0)
        
        data = tuple(d.to(args.device) for d in data)
        target = target.to(args.device)
          
        optimizer.zero_grad()
        
        outputs = model(data[0], data[1])
        # 或者：outputs = model(*data)
        
        loss = loss_fn(outputs[0], outputs[1], target)
        # 或者：loss = loss_fn(*outputs, target)
        
        losses.append(loss.data.item())

        loss.backward()
        optimizer.step()
        
        if batch_idx % log_interval == 0:
            print('Train: [{}/{} ({:.0f}%)]\tLoss: {:.6f} '.format(
                batch_idx * len(data[0]), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), np.mean(losses)))           
            
            total_loss += np.sum(losses)
            losses = []
            
    return total_loss/(batch_idx + 1)

def test_siamese_epoch(test_loader, model, loss_fn, args, log_interval=50):
    with torch.no_grad():
        model.eval()
        losses = []
        num_samples = 0
    
        for batch_idx, (data, target) in enumerate(test_loader):

            num_samples += data[0].size(0)
            data = tuple(d.to(args.device) for d in data)
            target = target.to(args.device)
            outputs = model(data[0], data[1])

            loss = loss_fn(outputs[0], outputs[1], target)
            losses.append(loss.data.item())
    
        return np.sum(losses)/(batch_idx + 1)

In [ ]:
start_epoch = 0

# 主训练循环
for epoch in range(start_epoch, args.num_epochs):

    # 训练阶段
    train_loss = train_siamese_epoch(siamese_train_loader, model, loss_fn, optimizer, args)
    message = 'Epoch: {}/{}. Train set: Average loss: {:.4f}'.format(
        epoch + 1, args.num_epochs, train_loss)
    
    # 测试/验证阶段
    test_loss = test_siamese_epoch(siamese_test_loader, model, loss_fn, args)
    
    message += '\nEpoch: {}/{}. Validation set: Average loss: {:.4f}'.format(epoch + 1, args.num_epochs,
                                                                             test_loss)
    print(message)


## 3.5 可视化


In [ ]:
train_embeddings_cl, train_labels_cl = extract_embeddings(train_loader, model, args)
plot_embeddings(train_embeddings_cl, train_labels_cl, title='Train embeddings (constrastive loss)')
test_embeddings_cl, test_labels_cl = extract_embeddings(test_loader, model, args)
plot_embeddings(test_embeddings_cl, test_labels_cl, title='Test embeddings (contrastive loss)')

为了比较两个向量 $x_1$ 和 $x_2$，我们可以用`余弦相似度`

$$\text{similarity}=\frac{x_1 \cdot x_2}{\max(\Vert x_1 \Vert _2 \cdot \Vert x_2 \Vert_2, \epsilon)}$$

另一种选择是欧氏距离。


为了在查询时节省计算量，我们可以预处理向量，做 L2 归一化。现在只需要点积就能完成比较


#### 练习
用 `numpy` 对 embedding 做 L2 归一化


In [ ]:
# 对 embedding 做 L2 归一化
test_embeddings_norm = ....

### 问题
为什么要归一化特征？


### 练习
现在写一个函数 `most_sim`：计算查询向量与数据集中所有向量的点积，提取 `topk` 个最相似向量的索引，放进一个元组列表里（


In [ ]:
def most_sim(x, emb, topk=6):
       return None

In [ ]:
test_images_raw = get_raw_images(test_loader)

In [ ]:
def launch_query(test_embeddings_norm, test_images_raw, query_id=None):
    query_id = random.randint(0, test_embeddings_norm.shape[0]) if query_id is None else query_id
    query_vector = test_embeddings_norm[query_id,:]

    print(f'query_id: {query_id} | query_embedding: {query_vector}')
    knns = most_sim(query_vector, test_embeddings_norm)
    knn_images = np.array([test_images_raw[x[0]] for x in knns ])
    
    title=['q: 1.0', f'1nn: {knns[1][1]:.3}', f'2nn: {knns[2][1]:.3}', 
           f'3nn: {knns[3][1]:.3}', f'4nn: {knns[4][1]:.3}', f'5nn: {knns[5][1]:.3}']
    show(torchvision.utils.make_grid(torch.from_numpy(knn_images)), title=title)
#     print(knns)
    

In [ ]:
for i in range(5):
    launch_query(test_embeddings_norm, test_images_raw)

# 三元组网络
我们将训练一个三元组网络，它接收一个锚点（anchor）、一个正样本（与锚点同类）和一个负样本（与锚点不同类）。目标是通过某个边际值，学到让锚点比负样本更接近正样本的 embedding。

![alt text](images/anchor_negative_positive.png "Source: FaceNet")
来源：[2] *Schroff, Florian, Dmitry Kalenichenko, and James Philbin. [Facenet: A unified embedding for face recognition and clustering.](https://arxiv.org/abs/1503.03832) CVPR 2015.*

**三元组损失**：   $L_{triplet}(x_a, x_p, x_n) = max(0, m +  \lVert f(x_a)-f(x_p)\rVert_2^2 - \lVert f(x_a)-f(x_n)\rVert_2^2$


## 4.1 架构
我们首先在 `EmbeddingNet` 之上定义三元组架构

### 练习

填写 `TripleNet` 的 forward 部分


In [ ]:
class TripletNet(nn.Module):
    def __init__(self, embedding_net):
        super(TripletNet, self).__init__()
        self.embedding_net = embedding_net

    def forward(self, x1, x2, x3):
        # 这里缺 3 行
        
        return output1, output2, output3

    def get_embedding(self, x):
        return self.embedding_net(x)

## 4.2 数据加载器
我们还需要调整数据加载器，让它取三元组的图像


In [ ]:
from torch.utils.data import Dataset
from torch.utils.data.sampler import BatchSampler
from PIL import Image

class TripletMNIST(Dataset):
    """
    Train: For each sample (anchor) randomly chooses a positive and negative samples
    Test: Creates fixed triplets for testing
    """

    def __init__(self, mnist_dataset):
        self.mnist_dataset = mnist_dataset
        self.train = self.mnist_dataset.train
        self.transform = self.mnist_dataset.transform

        if self.train:
            self.train_labels = self.mnist_dataset.train_labels
            self.train_data = self.mnist_dataset.train_data
            self.labels_set = set(self.train_labels.numpy())
            self.label_to_indices = {label: np.where(self.train_labels.numpy() == label)[0]
                                     for label in self.labels_set}

        else:
            self.test_labels = self.mnist_dataset.test_labels
            self.test_data = self.mnist_dataset.test_data
            # 生成固定的测试用三元组
            self.labels_set = set(self.test_labels.numpy())
            self.label_to_indices = {label: np.where(self.test_labels.numpy() == label)[0]
                                     for label in self.labels_set}

            random_state = np.random.RandomState(29)

            triplets = [[i,
                         random_state.choice(self.label_to_indices[self.test_labels[i].item()]),
                         random_state.choice(self.label_to_indices[
                                                 np.random.choice(
                                                     list(self.labels_set - set([self.test_labels[i].item()]))
                                                 )
                                             ])
                         ]
                        for i in range(len(self.test_data))]
            self.test_triplets = triplets

    def __getitem__(self, index):
        if self.train:
            img1, label1 = self.train_data[index], self.train_labels[index].item()
            positive_index = index
            while positive_index == index:
                positive_index = np.random.choice(self.label_to_indices[label1])
            negative_label = np.random.choice(list(self.labels_set - set([label1])))
            negative_index = np.random.choice(self.label_to_indices[negative_label])
            img2 = self.train_data[positive_index]
            img3 = self.train_data[negative_index]
        else:
            img1 = self.test_data[self.test_triplets[index][0]]
            img2 = self.test_data[self.test_triplets[index][1]]
            img3 = self.test_data[self.test_triplets[index][2]]

        img1 = Image.fromarray(img1.numpy(), mode='L')
        img2 = Image.fromarray(img2.numpy(), mode='L')
        img3 = Image.fromarray(img3.numpy(), mode='L')
        if self.transform is not None:
            img1 = self.transform(img1)
            img2 = self.transform(img2)
            img3 = self.transform(img3)
        return (img1, img2, img3), []

    def __len__(self):
        return len(self.mnist_dataset)

## 4.3 损失函数

### 练习

填写 `triplet loss` 中缺失的部分：
 $L_{triplet}(x_a, x_p, x_n) = max(0, m +  \lVert f(x_a)-f(x_p)\rVert_2^2 - \lVert f(x_a)-f(x_n)\rVert_2^2$


In [ ]:
class TripletLoss(nn.Module):
    """
    Triplet loss
    Takes embeddings of an anchor sample, a positive sample and a negative sample
    """

    def __init__(self, margin):
        super(TripletLoss, self).__init__()
        self.margin = margin

    def forward(self, anchor, positive, negative, size_average=True):
        distance_positive = None  # 填写代码
        distance_negative = None  # 填写代码
        # 你可以再次用 ReLU 代替 max
        losses = None  # 填写代码
        return losses.mean() if size_average else losses.sum()

## 4.4 训练


In [ ]:
triplet_train_dataset = TripletMNIST(train_dataset)  # 返回图像三元组
triplet_test_dataset = TripletMNIST(test_dataset)

args.batch_size = 128
kwargs = {'num_workers': args.num_workers, 'pin_memory': True} 
triplet_train_loader = torch.utils.data.DataLoader(triplet_train_dataset, batch_size=args.batch_size, shuffle=True, **kwargs)
triplet_test_loader = torch.utils.data.DataLoader(triplet_test_dataset, batch_size=args.batch_size, shuffle=False, **kwargs)

margin = 1.
embedding_net = EmbeddingNet()
model = TripletNet(embedding_net)
loss_fn = TripletLoss(margin)
model.to(args.device)
loss_fn.to(args.device)
args.lr = 1e-3
optimizer = optim.Adam(model.parameters(), lr=args.lr)
scheduler = lr_scheduler.StepLR(optimizer, 8, gamma=0.1, last_epoch=-1)
n_epochs = 5
log_interval = 100

### 练习

参照前面的例子，自己写训练/测试序列。
不过要注意一些差异。


In [ ]:
def train_triplet_epoch(train_loader, model, loss_fn, optimizer, args, log_interval=100):
    model.train()
    losses = []
    total_loss, num_samples =  0, 0
  
    # 在这里填写代码
    
    return total_loss/(batch_idx + 1)

def test_triplet_epoch(test_loader, model, loss_fn, args, log_interval=50):
    losses = []
    num_samples = 0
    # 在这里填写代码

    return np.sum(losses)/(batch_idx + 1)


In [ ]:
start_epoch = 0


# 主训练循环
for epoch in range(start_epoch, args.num_epochs):

    # 训练阶段
    train_loss = train_triplet_epoch(triplet_train_loader, model, loss_fn, optimizer, args)
    message = 'Epoch: {}/{}. Train set: Average loss: {:.4f}'.format(
        epoch + 1, args.num_epochs, train_loss)
    
    # 测试/验证阶段
    test_loss = test_triplet_epoch(triplet_test_loader, model, loss_fn, args)
    
    message += '\nEpoch: {}/{}. Validation set: Average loss: {:.4f}'.format(epoch + 1, args.num_epochs,
                                                                             test_loss)
    print(message)

## 4.5 可视化


In [ ]:
train_embeddings_tl, train_labels_tl = extract_embeddings(train_loader, model, args)
plot_embeddings(train_embeddings_tl, train_labels_tl, title='Train triplet embeddings')
test_embeddings_tl, test_labels_tl = extract_embeddings(test_loader, model, args)
plot_embeddings(test_embeddings_tl, test_labels_tl, title='Val triplet embeddings')

In [ ]:
# 对 embedding 做 L2 归一化
test_embeddings_tl_norm = test_embeddings_tl / np.linalg.norm(test_embeddings_tl, axis=-1, keepdims=True)

In [ ]:
test_images_raw = get_raw_images(test_loader)

In [ ]:
for i in range(5):
    launch_query(test_embeddings_tl_norm, test_images_raw)